# NOVA numerical audit: explicit inverse vs Cholesky solve

This notebook is a small, deterministic demonstration of the numerical trade-off in evaluating `d.T @ Sigma^-1 @ d`. It is a benchmark, not a proof of universal superiority, physical validity, or a third-party reproduction of a full CosmoSIS run.

The stress test reports residuals and failures over a controlled family of symmetric positive-definite matrices. Results can depend on BLAS/LAPACK and hardware, so the environment is recorded below.

In [ ]:
import platform, sys, time, hashlib
import numpy as np
import scipy
from scipy.linalg import cho_factor, cho_solve
import matplotlib.pyplot as plt

print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('numpy:', np.__version__)
print('scipy:', scipy.__version__)
print('seed: 20261119')

## 1. Two evaluation paths

The baseline forms an explicit inverse. The solve path factors `Sigma` and solves the linear system without materialising `Sigma^-1`. Both paths evaluate the same quadratic form.

In [ ]:
def quadratic_form_inverse(cov, delta):
    return float(delta @ np.linalg.inv(cov) @ delta)

def quadratic_form_cholesky(cov, delta):
    factor = cho_factor(cov, lower=True, check_finite=True)
    solved = cho_solve(factor, delta, check_finite=True)
    return float(delta @ solved)

def make_spd(condition_number, dimension=24, seed=20261119):
    rng = np.random.default_rng(seed)
    q, _ = np.linalg.qr(rng.normal(size=(dimension, dimension)))
    eig = np.geomspace(1.0, 1.0 / condition_number, dimension)
    return q @ np.diag(eig) @ q.T

def relative_residual(cov, delta, value):
    residual = abs(delta @ np.linalg.solve(cov, delta) - value)
    scale = max(abs(value), np.finfo(float).tiny)
    return residual / scale

In [ ]:
# Experiment 1: well-conditioned deterministic case
cov = make_spd(1e2, dimension=24)
delta = np.linspace(-1.0, 1.0, cov.shape[0])
q_inv = quadratic_form_inverse(cov, delta)
q_chol = quadratic_form_cholesky(cov, delta)
abs_diff = abs(q_inv - q_chol)
print(f'inverse value:   {q_inv:.17g}')
print(f'cholesky value:  {q_chol:.17g}')
print(f'absolute difference: {abs_diff:.3e}')
assert np.isfinite(q_inv) and np.isfinite(q_chol)
assert abs_diff < 1e-10
print('NORMAL_CASE: PASS')

## 2. Condition-number stress test

This does not assume that Cholesky remains accurate at every condition number. It measures both paths against `np.linalg.solve` and records exceptions or non-finite values. The conclusion is empirical for this matrix family and runtime only.

In [ ]:
condition_numbers = np.geomspace(1e2, 1e16, 15)
rows = []
for cond in condition_numbers:
    cov = make_spd(float(cond), dimension=24)
    delta = np.linspace(-1.0, 1.0, cov.shape[0])
    reference = float(delta @ np.linalg.solve(cov, delta))
    row = {'condition_number': float(cond), 'reference': reference}
    for name, fn in [('inverse', quadratic_form_inverse), ('cholesky', quadratic_form_cholesky)]:
        try:
            value = fn(cov, delta)
            row[name + '_relative_residual'] = relative_residual(cov, delta, value)
            row[name + '_finite'] = bool(np.isfinite(value))
            row[name + '_error'] = ''
        except Exception as exc:
            row[name + '_relative_residual'] = np.nan
            row[name + '_finite'] = False
            row[name + '_error'] = type(exc).__name__
    rows.append(row)

for row in rows:
    print(row)
print('STRESS_TEST: PASS (completed; inspect residuals and exceptions above)')

In [ ]:
x = [r['condition_number'] for r in rows]
plt.figure(figsize=(7, 4.5))
plt.loglog(x, [r['inverse_relative_residual'] for r in rows], 'o-', label='explicit inverse')
plt.loglog(x, [r['cholesky_relative_residual'] for r in rows], 'o-', label='Cholesky solve')
plt.xlabel('condition number')
plt.ylabel('relative residual')
plt.title('NOVA audit: inverse vs Cholesky solve')
plt.grid(True, which='both', alpha=0.25)
plt.legend()
plt.show()

## Reproduction report

Please report the Python/NumPy/SciPy versions, runtime, normal-case absolute difference, stress-test completion status, and any exceptions. Do not claim that this notebook proves universal numerical superiority or physical validity.